# Notebook 02 — EIS Feature Extraction
**Battery-PIAI-ECM v2.0.0**

Validates and summarises EIS-derived features:
- **Re**: electrolyte resistance (mΩ) — proxy for SEI layer growth
- **Rct**: charge-transfer resistance (mΩ) — proxy for kinetic degradation
- **Re–Rct coupling**: linear regression across all 636 cycles


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from src.config import PROCESSED_CSV, FIGURES_DIR
from src.eis_features import validate_eis, eis_summary, re_rct_coupling

df = pd.read_csv(PROCESSED_CSV)
df = validate_eis(df)
print(f'Loaded {len(df)} cycles')


## 2.1 Per-cell EIS summary


In [ ]:
summary = eis_summary(df)
print(summary.to_string(index=False))


## 2.2 Re–Rct coupling analysis
Strong positive coupling (R² > 0.93) confirms both parameters jointly track cell aging.


In [ ]:
coupling = re_rct_coupling(df)
print(f"Re–Rct: slope={coupling['slope']:.4f}, R²={coupling['R2']:.4f}, p={coupling['pval']:.2e}")

# Scatter plot
COLORS = {'B0005':'#2166AC','B0006':'#D6604D','B0007':'#4DAC26','B0018':'#8073AC'}
fig, ax = plt.subplots(figsize=(7, 5))
for bat, g in df.dropna(subset=['Re','Rct']).groupby('Battery'):
    ax.scatter(g['Re']*1000, g['Rct']*1000, s=12, alpha=0.6, color=COLORS[bat], label=bat)
valid = df.dropna(subset=['Re','Rct'])
sl, ic, rv, *_ = stats.linregress(valid['Re'], valid['Rct'])
x = np.linspace(valid['Re'].min(), valid['Re'].max(), 100)
ax.plot(x*1000, (sl*x+ic)*1000, 'k--', lw=2, label=f'Fit R²={rv**2:.3f}')
ax.set_xlabel('Re (mΩ)'); ax.set_ylabel('Rct (mΩ)')
ax.set_title('Re–Rct Coupling Across All NASA Cells')
ax.legend(); plt.tight_layout(); plt.show()


## 2.3 Re change over aging per cell


In [ ]:
for bat, g in df.groupby('Battery'):
    g = g.dropna(subset=['Re']).sort_values('CycleIndex')
    sl, ic, rv, *_ = stats.linregress(g['CycleIndex'], g['Re'])
    pct = (g.iloc[-1]['Re'] - g.iloc[0]['Re']) / g.iloc[0]['Re'] * 100
    print(f'{bat}: Re {g.iloc[0]["Re"]*1000:.1f}→{g.iloc[-1]["Re"]*1000:.1f} mΩ '
          f'({pct:+.1f}%), slope={sl*1e6:.2f}×10⁻⁶ Ω/cyc, R²={rv**2:.3f}')
